# Task 3 - Pin model experiments

Everything here is read from files written by `run_task3.py`; nothing is retrained in the notebook.

**Rules that keep the numbers honest**
- Parameters were fitted on *leave-one-out visit pseudo-labels* (`tune_pin_model.py`), never on surveyed truth.
- Surveyed truth enters only as neighbour knowledge from folds that are not being scored (see `test_task3.py`).
- Fold 0 is the official test group. It was exposed earlier in exploratory work (see the Task 1-2 README), so treat it as a sanity check, not a blind test.

In [1]:
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
R = Path('results')

## 1. Address parsing coverage

In [2]:
f = pd.read_csv(R/'address_features.csv')
cov = f[['pincode_text','loc_id','cross','main','gali','block','road','ward','building','lm_type']].notna().mean().round(3)
display(cov.rename('share of addresses with the field').to_frame())
display(f.lm_type.value_counts().rename('landmark type').to_frame().T)

,share of addresses with the field
pincode_text,0.919
loc_id,0.903
cross,0.267
main,0.266
gali,0.275
block,0.293
road,0.293
ward,0.083
building,0.076
lm_type,0.712


lm_type,govt_school,ganesha_temple,bus_stop,ration_shop,masjid,hanuman_temple,medical_store,milk_dairy,church,community_hall,water_tank,post_office,park,petrol_bunk
landmark type,192,190,190,171,169,160,159,156,138,138,121,96,88,84


## 2. What each layer adds (ablation ladder)

In [3]:
ab = pd.read_csv(R/'pin_model_ablation.csv')
display(ab.round(1))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
y = np.arange(len(ab))[::-1]
ax[0].barh(y, ab.all100_median, color='#3b6ea5'); ax[0].set_yticks(y); ax[0].set_yticklabels(ab.variant, fontsize=8)
ax[0].set_xlabel('median error, 100 surveyed (m)'); ax[0].set_title('Median')
ax[1].barh(y, ab.all100_p90, color='#c26a3d'); ax[1].set_yticks([]); ax[1].set_xlabel('90th percentile error (m)'); ax[1].set_title('P90')
plt.tight_layout(); plt.savefig(R/'pin_model_ablation.png', dpi=130); plt.show()

,variant,dev85_median,dev85_p90,test15_median,test15_p90,all100_median,all100_p90,within50,within100,within250
0,0 old geocoder only,362.9,747.7,455.5,2845.2,376.4,839.2,0.0,0.1,0.4
1,1 + own visit evidence,117.7,615.6,154.3,2477.0,125.9,625.1,0.4,0.5,0.6
2,2 + locality group/centroid,98.6,536.9,142.6,2577.8,118.8,550.0,0.4,0.5,0.6
3,3 + neighbours on same street key,68.7,534.1,69.4,2577.8,69.0,550.0,0.4,0.5,0.7
4,4 + street-number grid regression,33.3,500.5,58.3,2577.8,37.0,537.4,0.5,0.6,0.8
5,5 + landmark groups and POIs (full model),38.5,351.4,68.3,2472.6,45.7,368.6,0.5,0.6,0.9


C:\Users\ASUS\AppData\Local\Temp\ipykernel_26364\1405834359.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(R/'pin_model_ablation.png', dpi=130); plt.show()


## 3. Accuracy on the 100 surveyed addresses (out-of-fold)

`dev85` = development folds 1-5, `test15` = official-test fold 0. Surveyed addresses are about 49% visited / 51% unvisited, so the overall number mixes two very different populations - see section 4.

In [4]:
sc = pd.read_csv(R/'pin_model_scores.csv')
display(sc[['dimension','segment','n','median_error_m','median_ci95_low_m','median_ci95_high_m','p90_error_m','share_within_50m','share_within_100m','share_within_250m']].round(2))

,dimension,segment,n,median_error_m,median_ci95_low_m,median_ci95_high_m,p90_error_m,share_within_50m,share_within_100m,share_within_250m
0,overall,all,100,45.68,27.76,74.33,368.57,0.51,0.65,0.86
1,precision,locality,73,47.33,22.79,96.96,294.67,0.51,0.63,0.89
2,precision,pincode,10,884.82,9.17,2416.43,3963.04,0.40,0.40,0.40
3,precision,rooftop,1,25.59,NaN,NaN,25.59,1.00,1.00,1.00
4,precision,street,16,41.53,25.95,72.08,87.82,0.56,0.88,1.00
5,town_id,T1,33,34.14,22.79,70.90,334.40,0.58,0.70,0.85
6,town_id,T2,29,69.37,19.21,111.00,250.68,0.45,0.66,0.90
7,town_id,T3,38,51.75,19.29,112.31,618.78,0.50,0.61,0.84
8,split,test,15,68.27,28.24,161.10,2472.61,0.47,0.73,0.80
9,split,train,66,73.20,31.97,126.49,461.71,0.45,0.56,0.83


In [5]:
oof = pd.read_csv(R/'pin_model_oof_errors.csv')
base = pd.read_csv('clean_data/baseline_geocodes.csv').set_index('address_id'); sv = pd.read_csv('clean_data/surveyed_addresses.csv').set_index('address_id')
old = np.hypot(base.geocoder_x - sv.surveyed_x, base.geocoder_y - sv.surveyed_y).loc[oof.address_id].values
fig, ax = plt.subplots(figsize=(6.2, 4))
for e, lab, c in [(old, 'old geocoder', '#999999'), (oof.error_m.values, 'pin model (out-of-fold)', '#3b6ea5')]:
    s = np.sort(e); ax.plot(s, np.arange(1, len(s)+1)/len(s), label=lab, color=c, lw=2)
ax.set_xscale('log'); ax.set_xlabel('distance error (m, log scale)'); ax.set_ylabel('share of addresses'); ax.grid(alpha=.3); ax.legend()
ax.set_title('Error distribution, 100 surveyed addresses'); plt.tight_layout(); plt.savefig(R/'pin_model_error_cdf.png', dpi=130); plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26364\2330917303.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title('Error distribution, 100 surveyed addresses'); plt.tight_layout(); plt.savefig(R/'pin_model_error_cdf.png', dpi=130); plt.show()


## 4. The harder question: addresses with NO visit

The hide-and-predict check hides each visited address's own visit evidence and asks the model to recover the visit pin from text, neighbours, landmarks and the old pin. It uses 1,231 addresses and **no surveyed truth**, so it is both large and independent of the 100-address survey. This is the realistic number for the ~57% of addresses that have no usable visit.

In [6]:
hp = pd.read_csv(R/'hide_and_predict.csv')
tab = pd.DataFrame({'old geocoder': hp.err_old.describe(percentiles=[.5,.9])[['50%','90%']], 'pin model': hp.err_model.describe(percentiles=[.5,.9])[['50%','90%']]}).T
tab['within 100 m'] = [(hp.err_old<=100).mean(), (hp.err_model<=100).mean()]; display(tab.round(2))
display(hp.groupby('precision')[['err_old','err_model']].median().round(0).join(hp.groupby('precision').size().rename('n')))
display(hp.groupby('method_used').agg(n=('err_model','size'), model_median=('err_model','median'), old_median=('err_old','median')).round(0))

,50%,90%,within 100 m
old geocoder,320.54,797.19,0.12
pin model,126.33,606.91,0.42


,err_old,err_model,n
precision,,,
locality,354.0,139.0,891
pincode,1352.0,1369.0,94
rooftop,18.0,20.0,26
street,121.0,73.0,220


,n,model_median,old_median
method_used,,,
baseline,210,110.0,133.0
blend,64,82.0,250.0
landmark_group,79,172.0,367.0
landmark_poi,127,213.0,332.0
locality_centroid,93,1358.0,1356.0
street_grid,589,89.0,317.0
street_key,69,92.0,396.0


## 5. Is `raw_spread` honest? (input to Task 4's calibrated radii)

In [7]:
oofm = oof.merge(pd.read_csv('pins.csv')[['address_id','raw_spread','method_used']], on='address_id')
hp2 = hp.copy()
for name, e, s in [('surveyed OOF (n=%d)' % len(oofm), oofm.error_m, oofm.raw_spread), ('hide-and-predict (n=%d)' % len(hp2), hp2.err_model, hp2.raw_spread)]:
    r = e / s
    print(f'{name:30s} Spearman(raw_spread, error) = {pd.Series(s.values).corr(pd.Series(e.values), method="spearman"):.2f}; '
          f'share of errors within 1.18x / 2.15x raw_spread = {(r<=1.177).mean():.2f} / {(r<=2.146).mean():.2f} (Gaussian ideal 0.50 / 0.90)')
fig, ax = plt.subplots(figsize=(5.2, 4))
ax.scatter(hp2.raw_spread, hp2.err_model, s=6, alpha=.35, color='#3b6ea5'); lim = [10, 4000]; ax.plot(lim, lim, 'k--', lw=1, label='error = spread')
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('raw_spread (m)'); ax.set_ylabel('actual error (m)'); ax.legend(); ax.set_title('hide-and-predict'); plt.tight_layout(); plt.savefig(R/'pin_model_spread_vs_error.png', dpi=130); plt.show()

surveyed OOF (n=100)           Spearman(raw_spread, error) = 0.88; share of errors within 1.18x / 2.15x raw_spread = 0.85 / 0.96 (Gaussian ideal 0.50 / 0.90)
hide-and-predict (n=1231)      Spearman(raw_spread, error) = 0.69; share of errors within 1.18x / 2.15x raw_spread = 0.73 / 0.91 (Gaussian ideal 0.50 / 0.90)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_26364\2798940831.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('raw_spread (m)'); ax.set_ylabel('actual error (m)'); ax.legend(); ax.set_title('hide-and-predict'); plt.tight_layout(); plt.savefig(R/'pin_model_spread_vs_error.png', dpi=130); plt.show()


## 6. Where it fails

In [8]:
ad = pd.read_csv('clean_data/addresses.csv').set_index('address_id')
w = oofm.set_index('address_id').join(ad[['address_text']]).sort_values('error_m', ascending=False).head(10)
display(w[['precision','error_m','raw_spread','method_used','address_text']].round(0))

,precision,error_m,raw_spread,method_used,address_text
address_id,,,,,
AD001909,pincode,4740.0,1310.0,locality_centroid,#448 6th cr 8th mn kaveripura - 960104
AD001146,pincode,3877.0,2589.0,locality_centroid,"No. 122, nr Mosque, Navanagara East"
AD000170,locality,1791.0,820.0,baseline,"Blk F, Rd 6, Green Park Layout, Navanagara Eas..."
AD000141,pincode,1113.0,1321.0,locality_centroid,"Gali No. 4, Ward 1, Devgarh Nagar - 970202"
AD001786,locality,1087.0,768.0,landmark_group,"Sai Residency, Blk A, Rd 3, Doodh Dairy ke bag..."
AD001987,pincode,1052.0,1441.0,locality_centroid,"103/2, Gali No. 1, मस्जिद के पास, Devgarh Naga..."
AD001999,pincode,956.0,1301.0,locality_centroid,"#284, 3RD CROSS, 3RD MAIN, CLOSE TO POST OFFIC..."
AD002295,pincode,814.0,1737.0,locality_centroid,"Shanti Kunj, Blk A, Rd 4, Navanagara East - 98..."
AD001536,locality,535.0,817.0,baseline,no. 79 railway colony navanagara east - 980302


**Reading the failures.** The worst addresses are almost all *pincode-tier addresses whose text names no locality*. The pincode leaves 3-5 candidate localities about 1.4 km apart, nothing in the text separates them (street numbers repeat across localities; only T2 wards are locality-specific), and the old pin is already the pincode centroid. No text model can fix that; the model keeps close to the old pin there and reports a wide `raw_spread`, so the visit planner should send these addresses to *verify first*. One confirmed visit resolves them (the visit becomes the pin), which is the learning loop doing its job.

## 7. Limits to state honestly
- 100 surveyed addresses is small; intervals are in `pin_model_scores.csv`. Strata below 10 are not interpretable.
- The visit-derived labels used for tuning are not survey truth (they agree with it to a median of about 8 m where both exist).
- Surveyed truth covers residences only; office/native-village pins are unvalidated.
- The official-test fold was seen during exploratory analysis; freeze the method before claiming a blind result.